# Validación cruzada y costo operativo — SSD MC1

**Semana 10 · Ejercicio 3 · Antes de que falle**

Este cuaderno continúa el experimento de `ExperimentosModelos_MC1_Proyecto.ipynb`. Compara **la red neuronal MLP elegida como final** y **Gradient Boosting, segundo en PR-AUC de validación**, con dos esquemas de validación cruzada. También calcula una aproximación al costo de las falsas alarmas y de las fallas no detectadas.

La unidad de análisis es **SSD-día**, con `TARGET_30D = 1` si ocurre una falla en los 30 días posteriores. Las ventanas SMART de la OBT **ya están calculadas en Snowflake**. No se reconstruyen, no se hace imputación y nunca se incorporan `SSD_KEY`, fechas de falla ni `TARGET_30D` en las variables predictivas.

El notebook no modifica las tablas compartidas. Guarda archivos CSV locales y **no utiliza `display()`**. Los DataFrames aparecen como última expresión de las celdas para facilitar su exploración con Data Wrangler.

## 0. Requisitos y ubicación de archivos

Este notebook es **autónomo respecto de `pipeline_proyecto.py`**. Las funciones de muestreo, limpieza y división están definidas en la sección 0.2, sin depender de ese archivo.

Para recuperar los datos de Snowflake se sigue utilizando `snowflake_io.py`, el módulo de conexión con Spark del proyecto. Si ya tienes `mc1_muestra_experimentos.csv`, se reutiliza esa **muestra congelada** y no se hace una nueva consulta. También se recomienda disponer de `mc1_manifiesto_experimento.json`, generado durante la sesión anterior.

Reutiliza el servidor MLflow de tu contenedor mediante `MLFLOW_TRACKING_URI` (en Compose normalmente `http://mlflow:5000`). Este notebook **recupera los dos pipelines registrados y los clona para reentrenarlos en cada pliegue**. Cada pliegue se ajusta exclusivamente con sus observaciones de entrenamiento.

El entorno debe contar con scikit-learn, pandas, NumPy, MLflow y el módulo de conexión `snowflake_io` cuando se necesite consultar la OBT.


### 0.1. Importaciones

In [1]:
#? Bibliotecas del notebook y de las métricas.
import os
import json
import time
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import mlflow
import mlflow.sklearn
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    average_precision_score, fbeta_score,
    precision_score, recall_score, confusion_matrix,
)

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("MLflow:", mlflow.__version__)

Python: 3.10.12
scikit-learn: 1.7.2
MLflow: 2.17.2


### 0.2. Funciones auxiliares incorporadas en el notebook

Las funciones utilizadas anteriormente desde `pipeline_proyecto.py` se incluyen aquí para que **no sea necesario importar ese módulo externo**. Se organizan en cuatro grupos: configuración y huellas, consulta SQL, acceso a la muestra y preparación de particiones. Se preservan las reglas del experimento anterior y no se imputan mediciones SMART.


#### 0.2.1. Constantes, identificación de la OBT y huella SHA-256

Las 54 características corresponden a las combinaciones de atributos SMART, medidas temporales y ventanas de 7, 14 y 30 días. La huella permite verificar que el archivo utilizado sea el mismo entre ejecuciones.


In [2]:
#? Definir las constantes del proyecto y las funciones de identificación.
import hashlib
import re

SEMILLA = 42
OBJETIVO = "TARGET_30D"
SMART = ("R_1", "N_1", "R_5", "N_5", "R_9", "N_9")
VENTANAS = (7, 14, 30)
TIPOS = ("MEAN", "COUNT", "MEAN_SHIFT")
NUMERICAS = [f"{s}_{t}_{d}D" for s in SMART for d in VENTANAS for t in TIPOS]
PARTICIONES = ("ENTRENAMIENTO", "VALIDACION", "PRUEBA")
MAX_FILAS = 50_000

def sha256_archivo(ruta: str | Path) -> str:
    """Huella SHA-256 de los bytes de un archivo local."""
    h = hashlib.sha256()
    with open(ruta, "rb") as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()


def nombre_obt() -> str:
    """Nombre de la tabla compartida, sin credenciales dentro del código."""
    base = os.environ["SNOWFLAKE_DATABASE"].strip()
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_$]*", base):
        raise ValueError("SNOWFLAKE_DATABASE no es un identificador válido")
    return f'"{base}"."S_CDATOS_PSET2_OBT"."OBT_MC1_RN"'

print("Características SMART disponibles:", len(NUMERICAS))
print("Objetivo:", OBJETIVO)


Características SMART disponibles: 54
Objetivo: TARGET_30D


#### 0.2.2. Consulta de la muestra en Snowflake

La consulta respeta el embargo de julio y octubre de 2019. Aplica cuotas separadas a las dos clases del entrenamiento y mantiene el muestreo sin cuotas por clase para validación y prueba. La selección de identificadores SSD-día se realiza antes de recuperar las 54 características.


In [3]:
#? Construir el SQL sobre la OBT, sin materializar ni modificar tablas.
def construir_consulta(obt: str) -> str:
    """Reproduce el muestreo estratificado temporal establecido en el proyecto.

    Se seleccionan identificadores SSD-día antes de recuperar las 54 features.
    Las cuotas son máximas. El orden es determinista para una OBT fija.
    """
    columnas = ",\n        ".join(f"o.{c}" for c in NUMERICAS)
    return f"""
WITH candidatos AS (
    -- Fechas utilizables con etiquetas conocidas y embargos de julio y octubre.
    SELECT SSD_KEY, OBSERVATION_DATE, TARGET_30D,
           CASE
             WHEN OBSERVATION_DATE < '2019-07-01'::DATE THEN 'ENTRENAMIENTO'
             WHEN OBSERVATION_DATE BETWEEN '2019-08-01'::DATE AND '2019-09-30'::DATE THEN 'VALIDACION'
             WHEN OBSERVATION_DATE BETWEEN '2019-11-01'::DATE AND '2019-11-30'::DATE THEN 'PRUEBA'
           END AS PARTICION
    FROM {obt}
    WHERE TARGET_30D IN (0, 1)
      AND (OBSERVATION_DATE < '2019-07-01'::DATE
           OR OBSERVATION_DATE BETWEEN '2019-08-01'::DATE AND '2019-09-30'::DATE
           OR OBSERVATION_DATE BETWEEN '2019-11-01'::DATE AND '2019-11-30'::DATE)
),
reducidos AS (
    -- Conservar positivos y toda validación/prueba.
    -- Reducir de forma determinista los negativos de entrenamiento a ~5 %.
    SELECT * FROM candidatos
    WHERE PARTICION IS NOT NULL
      AND (TARGET_30D = 1 OR PARTICION IN ('VALIDACION', 'PRUEBA')
           OR MOD(MOD(HASH(SSD_KEY, OBSERVATION_DATE), 100) + 100, 100) < 5)
),
numerados AS (
    -- Dos rankings: por clase (entrenamiento) y por partición (val/prueba).
    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY PARTICION, TARGET_30D
             ORDER BY HASH(SSD_KEY, OBSERVATION_DATE), SSD_KEY, OBSERVATION_DATE
           ) AS FILA_CLASE,
           ROW_NUMBER() OVER (
             PARTITION BY PARTICION
             ORDER BY HASH(SSD_KEY, OBSERVATION_DATE), SSD_KEY, OBSERVATION_DATE
           ) AS FILA_PARTICION
    FROM reducidos
),
seleccion AS (
    -- Hasta 5 000 positivos + 25 000 negativos en entrenamiento.
    -- Hasta 10 000 de validación y 10 000 de prueba sin alterar sus clases.
    SELECT SSD_KEY, OBSERVATION_DATE, TARGET_30D, PARTICION
    FROM numerados
    WHERE (PARTICION = 'ENTRENAMIENTO'
           AND ((TARGET_30D = 1 AND FILA_CLASE <= 5000)
                OR (TARGET_30D = 0 AND FILA_CLASE <= 25000)))
       OR (PARTICION IN ('VALIDACION', 'PRUEBA') AND FILA_PARTICION <= 10000)
)
SELECT o.SSD_KEY, o.OBSERVATION_DATE,
       {columnas},
       s.TARGET_30D, s.PARTICION
FROM seleccion s
JOIN {obt} o
  ON o.SSD_KEY = s.SSD_KEY AND o.OBSERVATION_DATE = s.OBSERVATION_DATE
""".strip()

print("Consulta SQL de extracción preparada.")


Consulta SQL de extracción preparada.


#### 0.2.3. Consultar Snowflake y recuperar la muestra congelada

Si existe el CSV, se lee localmente. Cuando no existe o se solicita explícitamente actualizarlo, se realiza una consulta utilizando Spark y `snowflake_io.py`. El resultado se almacena como CSV para repetir la validación con las mismas observaciones.


In [4]:
#? Consultar mediante Spark, normalizar tipos y conservar el CSV de muestra.
def consultar_snowflake(sql: str, max_filas: int = MAX_FILAS) -> pd.DataFrame:
    """Replica la conexión Spark/Snowflake del notebook anterior."""
    from snowflake_io import create_spark_session, read_snowflake_query

    texto = sql.strip().rstrip(";")
    if not texto.upper().startswith(("SELECT", "WITH")) or ";" in texto:
        raise ValueError("La consulta debe ser SELECT o WITH, sin instrucciones adicionales")
    spark = create_spark_session("mc1-experimentos-mlflow")
    # El LIMIT exterior es un resguardo del equipo local, no el muestreo.
    resultado = read_snowflake_query(
        spark,
        f"SELECT * FROM ({texto}) q LIMIT {max_filas + 1}",
        schema="S_CDATOS_PSET2_OBT",
    ).toPandas()
    if len(resultado) > max_filas:
        raise ValueError(f"La consulta superó el máximo de {max_filas:,} observaciones")
    return resultado


def normalizar_tipos(datos: pd.DataFrame) -> pd.DataFrame:
    """Solo convierte tipos, sin imputación ni ingeniería adicional."""
    datos = datos.copy()
    requeridas = ["SSD_KEY", "OBSERVATION_DATE", *NUMERICAS, OBJETIVO, "PARTICION"]
    faltantes = sorted(set(requeridas) - set(datos.columns))
    if faltantes:
        raise ValueError(f"Faltan columnas de la OBT: {faltantes}")
    datos["SSD_KEY"] = datos["SSD_KEY"].astype("string")
    datos["OBSERVATION_DATE"] = pd.to_datetime(datos["OBSERVATION_DATE"], errors="raise")
    datos[NUMERICAS] = datos[NUMERICAS].apply(pd.to_numeric, errors="coerce")
    datos[NUMERICAS] = datos[NUMERICAS].replace([np.inf, -np.inf], np.nan)
    datos[OBJETIVO] = pd.to_numeric(datos[OBJETIVO], errors="raise").astype(int)
    if not set(datos[OBJETIVO].unique()).issubset({0, 1}):
        raise ValueError("Se encontró TARGET_30D fuera de 0/1")
    if not set(datos["PARTICION"].unique()).issubset(set(PARTICIONES)):
        raise ValueError("Partición desconocida")
    if datos.duplicated(["SSD_KEY", "OBSERVATION_DATE"]).any():
        raise ValueError("La OBT contiene pares SSD_KEY, OBSERVATION_DATE duplicados")
    return datos


def obtener_muestra(ruta_csv: str | Path, refrescar: bool = False) -> tuple[pd.DataFrame, str]:
    """Congela la muestra localmente y evita cambiarla accidentalmente entre corridas."""
    ruta = Path(ruta_csv)
    if ruta.exists() and not refrescar:
        datos = pd.read_csv(ruta, dtype={"SSD_KEY": "string"}, low_memory=False)
        fuente = "archivo_congelado"
    else:
        datos = consultar_snowflake(construir_consulta(nombre_obt()))
        datos = normalizar_tipos(datos)
        datos = datos.sort_values(
            ["PARTICION", "SSD_KEY", "OBSERVATION_DATE"], kind="stable"
        ).reset_index(drop=True)
        ruta.parent.mkdir(parents=True, exist_ok=True)
        datos.to_csv(ruta, index=False, date_format="%Y-%m-%d", float_format="%.17g")
        fuente = "snowflake"
    return normalizar_tipos(datos), fuente

print("Funciones de lectura y normalización disponibles.")


Funciones de lectura y normalización disponibles.


#### 0.2.4. Separar por fecha, seleccionar características y tratar los nulos

Estas funciones mantienen las particiones originales, usan exclusivamente entrenamiento para seleccionar columnas y eliminan filas incompletas en lugar de inventar valores SMART mediante imputación. Los dos modelos se evalúan con las mismas observaciones.


In [5]:
#? Dividir los períodos y preparar los conjuntos sin imputación.
def dividir_por_fecha(datos: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """Respeta la partición de SQL y verifica sus límites cronológicos."""
    particiones = {
        p: datos.loc[datos["PARTICION"] == p].copy()
        for p in PARTICIONES
    }
    if any(df.empty for df in particiones.values()):
        raise ValueError("Una partición quedó vacía")
    limites = {
        "ENTRENAMIENTO": (None, "2019-06-30"),
        "VALIDACION": ("2019-08-01", "2019-09-30"),
        "PRUEBA": ("2019-11-01", "2019-11-30"),
    }
    for nombre, conjunto in particiones.items():
        desde, hasta = limites[nombre]
        if desde and (conjunto["OBSERVATION_DATE"] < pd.Timestamp(desde)).any():
            raise ValueError(f"Fecha inválida en {nombre}")
        if (conjunto["OBSERVATION_DATE"] > pd.Timestamp(hasta)).any():
            raise ValueError(f"Fecha inválida en {nombre}")
        if set(conjunto[OBJETIVO].unique()) != {0, 1}:
            raise ValueError(f"La partición {nombre} no contiene las dos clases")
    return particiones


def seleccionar_caracteristicas(
    entrenamiento: pd.DataFrame, max_fraccion_nulos: float = 0.05
) -> tuple[list[str], pd.DataFrame]:
    """Filtra columnas usando SOLO entrenamiento: nulos y varianza nula."""
    analisis = pd.DataFrame({
        "caracteristica": NUMERICAS,
        "fraccion_nulos_entrenamiento": entrenamiento[NUMERICAS].isna().mean().values,
        "valores_distintos_entrenamiento": entrenamiento[NUMERICAS].nunique(dropna=True).values,
    })
    analisis["utilizar"] = (
        (analisis["fraccion_nulos_entrenamiento"] <= max_fraccion_nulos)
        & (analisis["valores_distintos_entrenamiento"] > 1)
    )
    elegidas = analisis.loc[analisis["utilizar"], "caracteristica"].tolist()
    if len(elegidas) < 2:
        raise ValueError("Quedaron menos de dos características. Revisar nulos")
    return elegidas, analisis


def preparar_conjuntos_sin_imputacion(
    particiones: dict[str, pd.DataFrame], columnas: list[str]
) -> tuple[dict[str, pd.DataFrame], pd.DataFrame]:
    """Elimina solo SSD-día incompletos en las columnas elegidas.

    Los cuatro modelos reciben las mismas filas por partición.
    No se inventan lecturas SMART para rellenar ventanas incompletas.
    """
    preparados = {}
    registros = []
    for nombre, datos in particiones.items():
        limpios = datos.dropna(subset=columnas).copy()
        if set(limpios[OBJETIVO].unique()) != {0, 1}:
            raise ValueError(f"La limpieza dejó una clase ausente en {nombre}")
        preparados[nombre] = limpios
        registros.append({
            "particion": nombre,
            "filas_antes": len(datos),
            "filas_despues": len(limpios),
            "filas_eliminadas": len(datos) - len(limpios),
            "positivos": int((limpios[OBJETIVO] == 1).sum()),
            "negativos": int((limpios[OBJETIVO] == 0).sum()),
            "ssd_positivos_unicos": int(limpios.loc[limpios[OBJETIVO] == 1, "SSD_KEY"].nunique()),
        })
    return preparados, pd.DataFrame(registros)

print("Funciones de división temporal y preparación disponibles.")


Funciones de división temporal y preparación disponibles.


### 0.3. Rutas y parámetros reproducibles

In [6]:
#? Parámetros para recuperar exactamente la muestra anterior.
DIRECTORIO = Path.cwd()
ARCHIVO_DATOS = DIRECTORIO / "mc1_muestra_experimentos.csv"
ARCHIVO_MANIFIESTO = DIRECTORIO / "mc1_manifiesto_experimento.json"

REFRESCAR_DESDE_SNOWFLAKE = False
MAX_FRACCION_NULOS = 0.05

#? Los IDs provienen de la ejecución documentada en el notebook anterior.
#? Solo cámbialos si volviste a entrenar y seleccionar otros modelos.
RUN_ID_FINAL = "7ed95ce1c1e5471db84ba174a1ab1ba3"
RUN_ID_SEGUNDO = "113da9a0fc62477ead058c67b0cdae5a"
NOMBRE_EXPERIMENTO = "Antes de que falle - SSD MC1 - TARGET_30D"

#? Costos RELATIVOS (hipótesis de aula, no gastos reales).
COSTO_FP = 1.0       # Alerta e inspección de una unidad sana.
COSTO_FN = 20.0      # No detectar un SSD que fallará.
UMBRAL = 0.50        # Se conserva fijo, sin optimizarlo con prueba.

PLIEGUES_ALEATORIOS = 3
print("Directorio:", DIRECTORIO)
print("Archivo de muestra:", ARCHIVO_DATOS.name)
print("Refrescar Snowflake:", REFRESCAR_DESDE_SNOWFLAKE)

Directorio: /opt/spark/notebooks
Archivo de muestra: mc1_muestra_experimentos.csv
Refrescar Snowflake: False


## 1. Recuperar la muestra de SSD-día

Se utiliza **la misma muestra de la sesión anterior**, sin volver a seleccionar SSDs ni introducir otra proporción de positivos. La extracción original tiene como máximos 5.000 positivos y 25.000 negativos en entrenamiento, 10.000 observaciones en validación y 10.000 en prueba. Solamente se consulta Snowflake si no existe el CSV o se activa explícitamente la actualización.

El entrenamiento de la sesión anterior fue sobremuestreado por clase. Por eso las métricas de validación cruzada **no representan directamente la prevalencia real**. Las particiones originales de agosto–septiembre y noviembre conservan un muestreo sin cuotas por clase.

### 1.1. Recuperar los registros

In [7]:
#? Reutilizar CSV congelado o consultar Snowflake mediante Spark.
datos, fuente_datos = obtener_muestra(
    ARCHIVO_DATOS,
    refrescar=REFRESCAR_DESDE_SNOWFLAKE,
)
print("Fuente:", fuente_datos)
print("Observaciones:", len(datos))
print("Columnas:", len(datos.columns))

Fuente: archivo_congelado
Observaciones: 50000
Columnas: 58


### 1.2. Inspeccionar la muestra

In [8]:
#? Datos originales disponibles en Data Wrangler.
datos.head(20)

,SSD_KEY,OBSERVATION_DATE,R_1_MEAN_7D,R_1_COUNT_7D,R_1_MEAN_SHIFT_7D,R_1_MEAN_14D,R_1_COUNT_14D,R_1_MEAN_SHIFT_14D,R_1_MEAN_30D,R_1_COUNT_30D,...,N_9_COUNT_7D,N_9_MEAN_SHIFT_7D,N_9_MEAN_14D,N_9_COUNT_14D,N_9_MEAN_SHIFT_14D,N_9_MEAN_30D,N_9_COUNT_30D,N_9_MEAN_SHIFT_30D,TARGET_30D,PARTICION
0,000035c97d82c585d0ae0a09dd82b7ed26d5f87f1c2e07...,2018-08-10,0.0,7,0.0,0.0,14,0.0,0.0,29,...,7,0.0,100,14,0.0,100,29,0.0,0,ENTRENAMIENTO
1,0004488a91d62006df7e4865aead4188a8a1d0a0d106f1...,2018-08-10,0.0,7,0.0,0.0,14,0.0,0.0,30,...,7,0.0,100,14,0.0,100,30,0.0,0,ENTRENAMIENTO
2,0005fa30cc4140cd15ae6d3ce893d70143318e021c162c...,2018-10-27,0.0,7,0.0,0.0,14,0.0,0.0,30,...,7,0.0,100,14,0.0,100,30,0.0,0,ENTRENAMIENTO
3,000ed4a53f671f0660d70daeefa70b5ab688c5038d2fca...,2018-04-03,0.0,7,0.0,0.0,14,0.0,0.0,30,...,7,0.0,100,14,0.0,100,30,0.0,0,ENTRENAMIENTO
4,0010224ca508a4c5f4fddfa0655c611c2b4f1859349eaa...,2019-02-09,0.0,7,0.0,0.0,14,0.0,0.0,30,...,7,0.0,100,14,0.0,100,30,0.0,0,ENTRENAMIENTO
5,00102dc1ef41ba97c6ab8d304926e11d209f5a8c8040f3...,2018-06-23,0.0,7,0.0,0.0,13,NaN,0.0,13,...,7,0.0,100,13,NaN,100,13,NaN,0,ENTRENAMIENTO
6,0014159c1a1e03eff88836223491150ae32a552eb8f2f3...,2018-12-18,0.0,7,0.0,0.0,14,0.0,0.0,30,...,7,0.0,100,14,0.0,100,30,0.0,0,ENTRENAMIENTO
7,001e40daecb696f53acc03d0282d9f1b249702d3c28acc...,2018-07-20,0.0,7,0.0,0.0,14,0.0,0.0,30,...,7,0.0,100,14,0.0,100,30,0.0,0,ENTRENAMIENTO
8,001e40daecb696f53acc03d0282d9f1b249702d3c28acc...,2018-09-15,0.0,7,0.0,0.0,14,0.0,0.0,30,...,7,0.0,100,14,0.0,100,30,0.0,0,ENTRENAMIENTO
9,001e6ca888ecac8d99451307355b448e3ad01813327898...,2018-11-22,0.0,7,0.0,0.0,10,0.0,0.0,26,...,7,0.0,100,10,0.0,100,26,0.0,0,ENTRENAMIENTO


### 1.3. Verificar el archivo y la procedencia

In [9]:
#? Registrar la huella de la muestra antes de validar.
huella_datos = sha256_archivo(ARCHIVO_DATOS)

manifiesto = (
    json.loads(ARCHIVO_MANIFIESTO.read_text(encoding="utf-8"))
    if ARCHIVO_MANIFIESTO.exists() else None
)
if manifiesto is not None:
    esperada = manifiesto.get("muestra_sha256")
    if esperada and esperada != huella_datos:
        raise ValueError("La muestra CSV no coincide con el manifiesto anterior.")

print("SHA-256:", huella_datos)
print("Manifiesto disponible:", manifiesto is not None)

SHA-256: 9fe1fcb4da4c4d099e4e989fdfbc7b7cb537f21d1f4e965aaf94bb64c42554ee
Manifiesto disponible: True


## 2. Recuperar exactamente las características del experimento anterior

La elección previa descarta atributos con más del 5 % de nulos en entrenamiento y columnas constantes. Se **reutiliza la lista exacta del manifiesto**, cuando existe, para garantizar que los modelos cargados desde MLflow reciban las mismas entradas y en el mismo orden. En ausencia del manifiesto se calcula esa lista únicamente a partir del conjunto original de entrenamiento.

No se imputan datos. Las filas incompletas en las columnas seleccionadas se excluyen igual que en la sesión anterior.

### 2.1. Separar las particiones originales

In [10]:
#? Mantener entrenamiento, validación y prueba del ejercicio anterior.
particiones_originales = dividir_por_fecha(datos)
print("Entrenamiento:", len(particiones_originales["ENTRENAMIENTO"]))
print("Validación:", len(particiones_originales["VALIDACION"]))
print("Prueba:", len(particiones_originales["PRUEBA"]))

Entrenamiento: 30000
Validación: 10000
Prueba: 10000


### 2.2. Recuperar las 41 características seleccionadas

In [11]:
#? Preferir columnas exactas del manifiesto, sin recalcularlas con el futuro.
if manifiesto is not None and manifiesto.get("caracteristicas_seleccionadas"):
    columnas_modelo = list(manifiesto["caracteristicas_seleccionadas"])
    origen_columnas = "manifiesto anterior"
else:
    columnas_modelo, _ = seleccionar_caracteristicas(
        particiones_originales["ENTRENAMIENTO"],
        max_fraccion_nulos=MAX_FRACCION_NULOS,
    )
    origen_columnas = "selección sobre entrenamiento original"

if len(columnas_modelo) != len(set(columnas_modelo)):
    raise ValueError("La lista de características contiene nombres repetidos.")
if not set(columnas_modelo).issubset(NUMERICAS):
    raise ValueError("Se encontraron características no reconocidas en la OBT.")

print("Origen de columnas:", origen_columnas)
print("Características utilizadas:", len(columnas_modelo))

Origen de columnas: manifiesto anterior
Características utilizadas: 41


### 2.3. Excluir registros incompletos sin imputación

In [12]:
#? Misma limpieza que el notebook de experimentos.
particiones_limpias, resumen_limpieza = preparar_conjuntos_sin_imputacion(
    particiones_originales, columnas_modelo
)
resumen_limpieza

,particion,filas_antes,filas_despues,filas_eliminadas,positivos,negativos,ssd_positivos_unicos
0,ENTRENAMIENTO,30000,29032,968,4916,24116,3190
1,VALIDACION,10000,9798,202,64,9734,64
2,PRUEBA,10000,9782,218,40,9742,39


### 2.4. Comprobar la distribución de clases

In [13]:
#? El desbalance real se aprecia en validación y prueba externas.
distribucion = pd.concat([
    pd.crosstab(par[OBJETIVO], pd.Series([nombre] * len(par), index=par.index))
    for nombre, par in particiones_limpias.items()
], axis=1).fillna(0).astype(int)
distribucion

col_0,ENTRENAMIENTO,VALIDACION,PRUEBA
TARGET_30D,,,
0,24116,9734,9742
1,4916,64,40


## 3. Preparar los datos de la validación cruzada

**Solo el conjunto `ENTRENAMIENTO` entra a los pliegues de validación cruzada.** Los meses de agosto–septiembre de 2019 y noviembre de 2019 permanecen fuera de ellos. La validación temporal se aplica sobre fechas reales, no sobre el orden arbitrario de las filas que resultaron del muestreo.

Un mismo SSD puede aparecer varias veces. Esto es coherente con el objetivo de predecir fallas futuras sobre una flota que incluye unidades conocidas, aunque no mide por separado la capacidad de generalizar a SSD completamente nuevos.

### 3.1. Ordenar por fecha y conservar la identidad SSD

In [14]:
#? Orden cronológico para construir los pliegues.
datos_cv = (
    particiones_limpias["ENTRENAMIENTO"]
    .sort_values(["OBSERVATION_DATE", "SSD_KEY"], kind="stable")
    .reset_index(drop=True)
)
X_cv = datos_cv[columnas_modelo].astype("float64")
y_cv = datos_cv[OBJETIVO].astype(int)
fechas_cv = datos_cv["OBSERVATION_DATE"]
ssd_cv = datos_cv["SSD_KEY"]

print("Observaciones CV:", len(datos_cv))
print("Características:", X_cv.shape[1])
print("Positivos:", int(y_cv.sum()))
print("Fechas:", fechas_cv.min().date(), "a", fechas_cv.max().date())

Observaciones CV: 29032
Características: 41
Positivos: 4916
Fechas: 2018-01-17 a 2019-06-30


### 3.2. Definir los períodos del esquema temporal

Se utilizan tres ventanas de evaluación situadas dentro del entrenamiento original. Cada pliegue se ajusta con el pasado, omite al menos 30 días inmediatamente anteriores al inicio de la ventana de evaluación y valida con las fechas posteriores. **El embargo se expresa en días calendario**, no en número de filas.

Esta división es de ventana expansiva: los conjuntos de entrenamiento crecen a medida que avanzan las fechas.

In [15]:
#? Cada período tiene un inicio y un fin de evaluación distintos.
PERIODOS_TEMPORALES = [
    ("temporal_1", "2018-09-01", "2018-10-31"),
    ("temporal_2", "2019-01-01", "2019-02-28"),
    ("temporal_3", "2019-05-01", "2019-06-30"),
]
DIAS_EMBARGO = 30

pd.DataFrame(PERIODOS_TEMPORALES, columns=["pliegue", "inicio_validacion", "fin_validacion"])

,pliegue,inicio_validacion,fin_validacion
0,temporal_1,2018-09-01,2018-10-31
1,temporal_2,2019-01-01,2019-02-28
2,temporal_3,2019-05-01,2019-06-30


### 3.3. Construir los índices temporales

In [16]:
#? Crear cada pliegue con entrenamiento ANTERIOR al embargo.
def construir_pliegues_temporales(fechas, periodos, dias_embargo=30):
    pliegues = []
    for nombre, inicio_txt, fin_txt in periodos:
        inicio = pd.Timestamp(inicio_txt)
        fin = pd.Timestamp(fin_txt)
        limite_entrenamiento = inicio - pd.Timedelta(days=dias_embargo)
        idx_ent = np.flatnonzero((fechas < limite_entrenamiento).to_numpy())
        idx_val = np.flatnonzero(((fechas >= inicio) & (fechas <= fin)).to_numpy())
        if len(idx_ent) == 0 or len(idx_val) == 0:
            raise ValueError(f"Pliegue vacío: {nombre}. Revisa sus fechas.")
        pliegues.append((nombre, idx_ent, idx_val))
    return pliegues

pliegues_temporales = construir_pliegues_temporales(
    fechas_cv, PERIODOS_TEMPORALES, DIAS_EMBARGO
)
print("Pliegues temporales:", len(pliegues_temporales))

Pliegues temporales: 3


### 3.4. Auditar fechas, clases y entidades por pliegue

In [17]:
#? Verificar que las dos clases estén disponibles en cada par entrenamiento/validación.
def describir_pliegues(pliegues, fechas, y, ssd, esquema):
    registros = []
    for nombre, idx_ent, idx_val in pliegues:
        y_ent = y.iloc[idx_ent]
        y_val = y.iloc[idx_val]
        if y_ent.nunique() != 2 or y_val.nunique() != 2:
            raise ValueError(
                f"{nombre} no contiene ambas clases. "
                "Ajusta las fechas y vuelve a auditar antes de entrenar."
            )
        fechas_ent = fechas.iloc[idx_ent]
        fechas_val = fechas.iloc[idx_val]
        if esquema == "temporal":
            assert fechas_ent.max() < fechas_val.min() - pd.Timedelta(days=DIAS_EMBARGO)
        registros.append({
            "esquema": esquema,
            "pliegue": nombre,
            "filas_entrenamiento": len(idx_ent),
            "filas_validacion": len(idx_val),
            "positivos_entrenamiento": int(y_ent.sum()),
            "positivos_validacion": int(y_val.sum()),
            "prevalencia_validacion": float(y_val.mean()),
            "ultima_fecha_entrenamiento": fechas_ent.max().date(),
            "primera_fecha_validacion": fechas_val.min().date(),
            "ssd_compartidos": len(set(ssd.iloc[idx_ent]) & set(ssd.iloc[idx_val])),
        })
    return pd.DataFrame(registros)

revision_temporal = describir_pliegues(
    pliegues_temporales, fechas_cv, y_cv, ssd_cv, "temporal"
)
revision_temporal

,esquema,pliegue,filas_entrenamiento,filas_validacion,positivos_entrenamiento,positivos_validacion,prevalencia_validacion,ultima_fecha_entrenamiento,primera_fecha_validacion,ssd_compartidos
0,temporal,temporal_1,8559,3395,854,342,0.100736,2018-08-01,2018-09-01,126
1,temporal,temporal_2,15207,3402,1697,458,0.134627,2018-12-01,2019-01-01,222
2,temporal,temporal_3,22188,4844,2665,1835,0.378819,2019-03-31,2019-05-01,458


### 3.5. Construir el esquema aleatorio de comparación

`StratifiedKFold` mezcla observaciones y mantiene aproximadamente la misma proporción de fallas en cada pliegue. **No respeta las fechas** ni mantiene separados los registros de un mismo SSD, por lo que puede ofrecer una evaluación optimista. Se incluye únicamente como contraste solicitado en la actividad.

In [18]:
#? Utilizar una semilla fija para repetir exactamente los pliegues aleatorios.
cv_aleatoria = StratifiedKFold(
    n_splits=PLIEGUES_ALEATORIOS,
    shuffle=True,
    random_state=SEMILLA,
)
pliegues_aleatorios = [
    (f"aleatorio_{i}", idx_ent, idx_val)
    for i, (idx_ent, idx_val) in enumerate(cv_aleatoria.split(X_cv, y_cv), 1)
]
print("Pliegues aleatorios:", len(pliegues_aleatorios))

Pliegues aleatorios: 3


### 3.6. Auditar los pliegues aleatorios

In [19]:
#? Contrastar cantidades, prevalencia y SSD repetidos.
revision_aleatoria = describir_pliegues(
    pliegues_aleatorios, fechas_cv, y_cv, ssd_cv, "aleatorio"
)
revision_aleatoria

,esquema,pliegue,filas_entrenamiento,filas_validacion,positivos_entrenamiento,positivos_validacion,prevalencia_validacion,ultima_fecha_entrenamiento,primera_fecha_validacion,ssd_compartidos
0,aleatorio,aleatorio_1,19354,9678,3277,1639,0.169353,2019-06-30,2018-01-17,1428
1,aleatorio,aleatorio_2,19355,9677,3278,1638,0.169267,2019-06-30,2018-01-17,1406
2,aleatorio,aleatorio_3,19355,9677,3277,1639,0.169371,2019-06-30,2018-01-17,1460


## 4. Recuperar los modelos de la sesión anterior desde MLflow

Los identificadores están fijados a la ejecución que figura en tu notebook anterior: **MLP como modelo final** y **Gradient Boosting como segundo mejor** en PR-AUC de validación. El notebook comprueba que la huella de la muestra coincide con la huella registrada. Si cambiaste las corridas, modifica sus `RUN_ID` en la sección 0.3.

Se recuperan los pipelines entrenados como referencia, pero cada pliegue empleará **`clone(pipeline)` y `fit()` exclusivamente sobre su propio entrenamiento**. Esto evita reutilizar estadísticas de escalado o coeficientes aprendidos usando observaciones del pliegue de validación.

### 4.1. Conectar al servidor de seguimiento

In [20]:
#? Reutilizar el tracking configurado en Docker Compose.
uri_mlflow = os.environ.get("MLFLOW_TRACKING_URI", "http://mlflow:5000")
mlflow.set_tracking_uri(uri_mlflow)
experimento = mlflow.get_experiment_by_name(NOMBRE_EXPERIMENTO)
if experimento is None:
    raise ValueError("No se encontró el experimento de la sesión anterior en MLflow.")

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experimento:", experimento.name)

Tracking URI: http://mlflow:5000
Experimento: Antes de que falle - SSD MC1 - TARGET_30D


### 4.2. Leer metadatos de las corridas existentes

In [21]:
#? No cambiar el ranking ni reentrenar los modelos en este paso.
ids_modelo = {"red_neuronal_mlp": RUN_ID_FINAL,
              "gradient_boosting": RUN_ID_SEGUNDO}
metadatos_modelos = {}
for nombre, rid in ids_modelo.items():
    run = mlflow.get_run(rid)
    if str(run.info.experiment_id) != str(experimento.experiment_id):
        raise ValueError(f"La corrida {rid} pertenece a otro experimento.")
    if run.info.status != "FINISHED":
        raise ValueError(f"La corrida {rid} no terminó correctamente.")
    huella_registrada = run.data.params.get("datos_sha256")
    if not huella_registrada or huella_registrada != huella_datos:
        raise ValueError(
            f"La huella de {nombre} no coincide con el CSV: "
            "no se deben mezclar muestras de dos experimentos."
        )
    if int(run.data.params.get("cantidad_features", -1)) != len(columnas_modelo):
        raise ValueError(f"Cambió el número de características de {nombre}.")
    metadatos_modelos[nombre] = run

resumen_original = pd.DataFrame([
    {"modelo": nombre, "run_id": run.info.run_id,
     "pr_auc_validacion_anterior": run.data.metrics.get("validacion_pr_auc"),
     "pr_auc_prueba_anterior": run.data.metrics.get("prueba_pr_auc")}
    for nombre, run in metadatos_modelos.items()
])
resumen_original

,modelo,run_id,pr_auc_validacion_anterior,pr_auc_prueba_anterior
0,red_neuronal_mlp,7ed95ce1c1e5471db84ba174a1ab1ba3,0.039071,0.013095
1,gradient_boosting,113da9a0fc62477ead058c67b0cdae5a,0.023344,0.025542


### 4.3. Descargar los pipelines entrenados

In [22]:
#? Recuperar el artefacto MLflow de cada corrida.
pipelines_originales = {
    nombre: mlflow.sklearn.load_model(f"runs:/{run.info.run_id}/pipeline_entrenado")
    for nombre, run in metadatos_modelos.items()
}
for nombre, pipeline in pipelines_originales.items():
    print(nombre, "->", list(pipeline.named_steps))

red_neuronal_mlp -> ['escalar', 'modelo']
gradient_boosting -> ['modelo']


### 4.4. Verificar la compatibilidad de las características

In [23]:
#? Comparar los nombres y su orden antes de cualquier entrenamiento.
for nombre, pipeline in pipelines_originales.items():
    nombres_entrenados = getattr(pipeline, "feature_names_in_", None)
    if nombres_entrenados is not None:
        if list(nombres_entrenados) != columnas_modelo:
            raise ValueError(f"El orden de características cambió en {nombre}.")
    print(nombre, ":", len(columnas_modelo), "características compatibles")

red_neuronal_mlp : 41 características compatibles
gradient_boosting : 41 características compatibles


## 5. Definir las métricas

La métrica técnica principal continúa siendo **PR-AUC, calculada con `average_precision_score`**. Este nombre se conserva para coincidir con el notebook anterior. Como complemento se calcula **F0.5**, que da más peso a la precisión que al recall.

Para representar un costo aproximado se supone que **una falla no detectada (FN) cuesta 20 veces más que una falsa alarma (FP)**. Son *unidades relativas* elegidas para la práctica y pueden modificarse en 0.3. El costo se calcula por **1.000 predicciones SSD-día**, no por 1.000 SSD-mes ni por eventos únicos. Como un SSD puede aportar varios días, este resultado **no es un costo real de operación**.

Las predicciones de clase utilizan un **umbral fijo de 0,50**, igual para ambos modelos y sin optimizarlo sobre los datos de prueba. Las probabilidades pueden estar mal calibradas por el sobremuestreo del entrenamiento.

### 5.1. Función de evaluación técnica y de costo

In [24]:
#? Convertir probabilidades y etiquetas en métricas comparables.
def calcular_metricas(y_verdadero, probabilidad, umbral=0.5,
                      costo_fp=1.0, costo_fn=20.0):
    y_real = np.asarray(y_verdadero, dtype=int)
    p = np.asarray(probabilidad, dtype=float)
    y_pred = (p >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_real, y_pred, labels=[0, 1]).ravel()
    costo_total = costo_fp * fp + costo_fn * fn
    return {
        "pr_auc": float(average_precision_score(y_real, p)),
        "f05": float(fbeta_score(y_real, y_pred, beta=0.5, zero_division=0)),
        "precision": float(precision_score(y_real, y_pred, zero_division=0)),
        "recall": float(recall_score(y_real, y_pred, zero_division=0)),
        "fp": int(fp), "fn": int(fn), "tp": int(tp), "tn": int(tn),
        "costo_total_relativo": float(costo_total),
        "costo_por_1000": float(1000.0 * costo_total / len(y_real)),
        "prevalencia": float(y_real.mean()),
    }

print("Costo FP:", COSTO_FP, "| Costo FN:", COSTO_FN)
print("Umbral:", UMBRAL)

Costo FP: 1.0 | Costo FN: 20.0
Umbral: 0.5


## 6. Ejecutar la validación cruzada

Cada entrenamiento empieza desde cero utilizando `clone()` sobre el pipeline recuperado. El escalador de la MLP se ajusta **solo en las filas de entrenamiento del pliegue**. Gradient Boosting utiliza las mismas columnas sin escalado. No se realizan imputaciones ni se calculan ventanas SMART nuevas.

### 6.1. Función reutilizable para los pliegues

In [25]:
#? Reentrenar cada modelo exclusivamente sobre los índices de su pliegue.
def evaluar_validacion_cruzada(nombre_modelo, pipeline_referencia, pliegues, esquema):
    registros = []
    for nombre_pliegue, idx_ent, idx_val in pliegues:
        estimador = clone(pipeline_referencia)  # Elimina parámetros previamente aprendidos.
        inicio = time.perf_counter()
        estimador.fit(X_cv.iloc[idx_ent], y_cv.iloc[idx_ent])
        segundos = time.perf_counter() - inicio
        probabilidades = estimador.predict_proba(X_cv.iloc[idx_val])[:, 1]
        metricas = calcular_metricas(
            y_cv.iloc[idx_val], probabilidades,
            umbral=UMBRAL, costo_fp=COSTO_FP, costo_fn=COSTO_FN,
        )
        registros.append({
            "modelo": nombre_modelo,
            "esquema": esquema,
            "pliegue": nombre_pliegue,
            "filas_entrenamiento": len(idx_ent),
            "filas_validacion": len(idx_val),
            "positivos_validacion": int(y_cv.iloc[idx_val].sum()),
            "segundos_entrenamiento": segundos,
            **metricas,
        })
        print(f"{esquema} / {nombre_modelo} / {nombre_pliegue}: "
              f"AP={metricas['pr_auc']:.5f}, "
              f"F0.5={metricas['f05']:.5f}, "
              f"costo/1000={metricas['costo_por_1000']:.2f}")
    return pd.DataFrame(registros)

### 6.2. Validación cruzada temporal — MLP

In [26]:
#? Primera evaluación: MLP en tres ventanas históricas con embargo.
cv_temporal_mlp = evaluar_validacion_cruzada(
    "red_neuronal_mlp", pipelines_originales["red_neuronal_mlp"],
    pliegues_temporales, "temporal"
)
cv_temporal_mlp

/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (120) reached and the optimization hasn't converged yet.
  warnings.warn(


temporal / red_neuronal_mlp / temporal_1: AP=0.52419, F0.5=0.62868, costo/1000=1272.75


/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (120) reached and the optimization hasn't converged yet.
  warnings.warn(


temporal / red_neuronal_mlp / temporal_2: AP=0.20922, F0.5=0.23035, costo/1000=2503.23
temporal / red_neuronal_mlp / temporal_3: AP=0.61952, F0.5=0.23965, costo/1000=7123.25


/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (120) reached and the optimization hasn't converged yet.
  warnings.warn(


,modelo,esquema,pliegue,filas_entrenamiento,filas_validacion,positivos_validacion,segundos_entrenamiento,pr_auc,f05,precision,recall,fp,fn,tp,tn,costo_total_relativo,costo_por_1000,prevalencia
0,red_neuronal_mlp,temporal,temporal_1,8559,3395,342,0.796563,0.524194,0.628684,0.757396,0.374269,41,214,128,3012,4321.0,1272.754050,0.100736
1,red_neuronal_mlp,temporal,temporal_2,15207,3402,458,1.459944,0.209222,0.230352,0.485714,0.074236,36,424,34,2908,8516.0,2503.233392,0.134627
2,red_neuronal_mlp,temporal,temporal_3,22188,4844,1835,2.067549,0.619520,0.239651,0.956522,0.059946,5,1725,110,3004,34505.0,7123.245252,0.378819


### 6.3. Validación cruzada temporal — Gradient Boosting

In [27]:
#? Segunda evaluación: Gradient Boosting en las mismas ventanas.
cv_temporal_boosting = evaluar_validacion_cruzada(
    "gradient_boosting", pipelines_originales["gradient_boosting"],
    pliegues_temporales, "temporal"
)
cv_temporal_boosting

temporal / gradient_boosting / temporal_1: AP=0.59941, F0.5=0.72125, costo/1000=1149.63
temporal / gradient_boosting / temporal_2: AP=0.20469, F0.5=0.21709, costo/1000=2519.99
temporal / gradient_boosting / temporal_3: AP=0.61278, F0.5=0.26993, costo/1000=7049.13


,modelo,esquema,pliegue,filas_entrenamiento,filas_validacion,positivos_validacion,segundos_entrenamiento,pr_auc,f05,precision,recall,fp,fn,tp,tn,costo_total_relativo,costo_por_1000,prevalencia
0,gradient_boosting,temporal,temporal_1,8559,3395,342,0.635811,0.599414,0.721248,0.865497,0.432749,23,194,148,3030,3903.0,1149.631811,0.100736
1,gradient_boosting,temporal,temporal_2,15207,3402,458,0.241899,0.204687,0.217087,0.484375,0.067686,33,427,31,2911,8573.0,2519.988242,0.134627
2,gradient_boosting,temporal,temporal_3,22188,4844,1835,0.264820,0.612780,0.269928,0.955224,0.069755,6,1707,128,3003,34146.0,7049.132948,0.378819


### 6.4. Validación cruzada aleatoria — MLP

In [28]:
#? Tercera evaluación: MLP con pliegues mezclados y estratificados.
cv_aleatoria_mlp = evaluar_validacion_cruzada(
    "red_neuronal_mlp", pipelines_originales["red_neuronal_mlp"],
    pliegues_aleatorios, "aleatorio"
)
cv_aleatoria_mlp

/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (120) reached and the optimization hasn't converged yet.
  warnings.warn(


aleatorio / red_neuronal_mlp / aleatorio_1: AP=0.53383, F0.5=0.54494, costo/1000=2386.03


/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (120) reached and the optimization hasn't converged yet.
  warnings.warn(


aleatorio / red_neuronal_mlp / aleatorio_2: AP=0.52866, F0.5=0.54486, costo/1000=2380.39
aleatorio / red_neuronal_mlp / aleatorio_3: AP=0.55600, F0.5=0.56007, costo/1000=2435.47


/usr/local/lib/python3.10/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (120) reached and the optimization hasn't converged yet.
  warnings.warn(


,modelo,esquema,pliegue,filas_entrenamiento,filas_validacion,positivos_validacion,segundos_entrenamiento,pr_auc,f05,precision,recall,fp,fn,tp,tn,costo_total_relativo,costo_por_1000,prevalencia
0,red_neuronal_mlp,aleatorio,aleatorio_1,19354,9678,1639,1.906034,0.533833,0.544935,0.681319,0.302624,232,1143,496,7807,23092.0,2386.030172,0.169353
1,red_neuronal_mlp,aleatorio,aleatorio_2,19355,9677,1638,1.692361,0.528663,0.544858,0.679400,0.304029,235,1140,498,7804,23035.0,2380.386483,0.169267
2,red_neuronal_mlp,aleatorio,aleatorio_3,19355,9677,1639,1.737205,0.556003,0.560067,0.736264,0.286150,168,1170,469,7870,23568.0,2435.465537,0.169371


### 6.5. Validación cruzada aleatoria — Gradient Boosting

In [29]:
#? Cuarta evaluación: Gradient Boosting sobre los mismos pliegues aleatorios.
cv_aleatoria_boosting = evaluar_validacion_cruzada(
    "gradient_boosting", pipelines_originales["gradient_boosting"],
    pliegues_aleatorios, "aleatorio"
)
cv_aleatoria_boosting

aleatorio / gradient_boosting / aleatorio_1: AP=0.54853, F0.5=0.56331, costo/1000=2452.06
aleatorio / gradient_boosting / aleatorio_2: AP=0.55853, F0.5=0.56916, costo/1000=2367.57
aleatorio / gradient_boosting / aleatorio_3: AP=0.57612, F0.5=0.58417, costo/1000=2372.53


,modelo,esquema,pliegue,filas_entrenamiento,filas_validacion,positivos_validacion,segundos_entrenamiento,pr_auc,f05,precision,recall,fp,fn,tp,tn,costo_total_relativo,costo_por_1000,prevalencia
0,gradient_boosting,aleatorio,aleatorio_1,19354,9678,1639,0.598836,0.548530,0.563311,0.752864,0.280659,151,1179,460,7888,23731.0,2452.056210,0.169353
1,gradient_boosting,aleatorio,aleatorio_2,19355,9677,1638,0.345097,0.558525,0.569161,0.724387,0.306471,191,1136,502,7848,22911.0,2367.572595,0.169267
2,gradient_boosting,aleatorio,aleatorio_3,19355,9677,1639,0.339355,0.576117,0.584172,0.758359,0.304454,159,1140,499,7879,22959.0,2372.532810,0.169371


## 7. Resumir promedio, desviación y resultados por pliegue

Se conservan todas las observaciones de los experimentos individuales en una tabla y se calcula un resumen por modelo y esquema. La desviación estándar usa `ddof=1`, adecuada para describir la variación entre los tres pliegues de esta práctica. Las medias temporales y aleatorias pueden diferir por cambios en las fechas, la dificultad y la prevalencia de las fallas, no solo por fuga de información.

### 7.1. Consolidar las doce evaluaciones

In [30]:
#? Una fila por modelo, esquema y pliegue.
resultados_pliegues = pd.concat([
    cv_temporal_mlp, cv_temporal_boosting,
    cv_aleatoria_mlp, cv_aleatoria_boosting,
], ignore_index=True)
resultados_pliegues

,modelo,esquema,pliegue,filas_entrenamiento,filas_validacion,positivos_validacion,segundos_entrenamiento,pr_auc,f05,precision,recall,fp,fn,tp,tn,costo_total_relativo,costo_por_1000,prevalencia
0,red_neuronal_mlp,temporal,temporal_1,8559,3395,342,0.796563,0.524194,0.628684,0.757396,0.374269,41,214,128,3012,4321.0,1272.754050,0.100736
1,red_neuronal_mlp,temporal,temporal_2,15207,3402,458,1.459944,0.209222,0.230352,0.485714,0.074236,36,424,34,2908,8516.0,2503.233392,0.134627
2,red_neuronal_mlp,temporal,temporal_3,22188,4844,1835,2.067549,0.619520,0.239651,0.956522,0.059946,5,1725,110,3004,34505.0,7123.245252,0.378819
3,gradient_boosting,temporal,temporal_1,8559,3395,342,0.635811,0.599414,0.721248,0.865497,0.432749,23,194,148,3030,3903.0,1149.631811,0.100736
4,gradient_boosting,temporal,temporal_2,15207,3402,458,0.241899,0.204687,0.217087,0.484375,0.067686,33,427,31,2911,8573.0,2519.988242,0.134627
5,gradient_boosting,temporal,temporal_3,22188,4844,1835,0.264820,0.612780,0.269928,0.955224,0.069755,6,1707,128,3003,34146.0,7049.132948,0.378819
6,red_neuronal_mlp,aleatorio,aleatorio_1,19354,9678,1639,1.906034,0.533833,0.544935,0.681319,0.302624,232,1143,496,7807,23092.0,2386.030172,0.169353
7,red_neuronal_mlp,aleatorio,aleatorio_2,19355,9677,1638,1.692361,0.528663,0.544858,0.679400,0.304029,235,1140,498,7804,23035.0,2380.386483,0.169267
8,red_neuronal_mlp,aleatorio,aleatorio_3,19355,9677,1639,1.737205,0.556003,0.560067,0.736264,0.286150,168,1170,469,7870,23568.0,2435.465537,0.169371
9,gradient_boosting,aleatorio,aleatorio_1,19354,9678,1639,0.598836,0.548530,0.563311,0.752864,0.280659,151,1179,460,7888,23731.0,2452.056210,0.169353


### 7.2. Calcular estadísticas agregadas

In [31]:
#? Promedio y desviación de AP, F0.5 y costo por 1000 predicciones.
resumen_cv = (
    resultados_pliegues
    .groupby(["esquema", "modelo"], as_index=False)
    .agg(
        pliegues=("pliegue", "count"),
        pr_auc_promedio=("pr_auc", "mean"),
        pr_auc_desviacion=("pr_auc", "std"),
        f05_promedio=("f05", "mean"),
        f05_desviacion=("f05", "std"),
        costo_1000_promedio=("costo_por_1000", "mean"),
        costo_1000_desviacion=("costo_por_1000", "std"),
        prevalencia_promedio=("prevalencia", "mean"),
    )
    .sort_values(["esquema", "pr_auc_promedio"], ascending=[True, False])
    .reset_index(drop=True)
)
resumen_cv

,esquema,modelo,pliegues,pr_auc_promedio,pr_auc_desviacion,f05_promedio,f05_desviacion,costo_1000_promedio,costo_1000_desviacion,prevalencia_promedio
0,aleatorio,gradient_boosting,3,0.561057,0.013966,0.572215,0.010761,2397.387205,47.409662,0.169330
1,aleatorio,red_neuronal_mlp,3,0.539500,0.014524,0.549953,0.008759,2400.627397,30.302389,0.169330
2,temporal,gradient_boosting,3,0.472294,0.231850,0.402754,0.277086,3572.917667,3087.478534,0.204727
3,temporal,red_neuronal_mlp,3,0.450979,0.214724,0.366229,0.227340,3633.077565,3084.553893,0.204727


### 7.3. Guardar los resultados para entregar

In [32]:
#? Exportar los pliegues y sus estadísticas como CSV locales.
RUTA_PLIEGUES = DIRECTORIO / "validacion_pliegues_proyecto.csv"
RUTA_RESUMEN = DIRECTORIO / "validacion_resumen_proyecto.csv"
resultados_pliegues.to_csv(RUTA_PLIEGUES, index=False)
resumen_cv.to_csv(RUTA_RESUMEN, index=False)
print("Guardado:", RUTA_PLIEGUES.name)
print("Guardado:", RUTA_RESUMEN.name)

Guardado: validacion_pliegues_proyecto.csv
Guardado: validacion_resumen_proyecto.csv


## 8. Calcular la métrica de negocio en las particiones externas

El esquema temporal se contrasta con las particiones de **agosto–septiembre** y **noviembre**, que conservan aproximadamente el desbalance real. Para esta sección **no se reentrena ningún modelo**: se utilizan los dos pipelines ajustados en la sesión pasada, recuperados desde MLflow.

La evaluación externa es distinta de la validación cruzada. Aquí el costo relativo por 1.000 SSD-día es más interpretable porque estas particiones no fueron balanceadas intencionalmente. Aun así, no corresponde directamente al costo de reemplazar o reparar SSD, porque cuenta predicciones diarias en lugar de episodios de alerta.

### 8.1. Separar los conjuntos externos

In [33]:
#? Preparar solo X/y, sin volver a ajustar nada sobre datos futuros.
X_externo = {}
y_externo = {}
for nombre in ["VALIDACION", "PRUEBA"]:
    df = particiones_limpias[nombre]
    X_externo[nombre] = df[columnas_modelo].astype("float64")
    y_externo[nombre] = df[OBJETIVO].astype(int)
    print(nombre, X_externo[nombre].shape, "positivos:", int(y_externo[nombre].sum()))

VALIDACION (9798, 41) positivos: 64
PRUEBA (9782, 41) positivos: 40


### 8.2. Evaluar la MLP entrenada sobre ambas particiones

In [34]:
#? Medir errores de clase y costo utilizando el modelo recuperado.
def evaluar_modelo_externo(nombre_modelo, pipeline, particion):
    prob = pipeline.predict_proba(X_externo[particion])[:, 1]
    metricas = calcular_metricas(
        y_externo[particion], prob,
        umbral=UMBRAL, costo_fp=COSTO_FP, costo_fn=COSTO_FN,
    )
    return {"modelo": nombre_modelo, "particion": particion,
            "observaciones": len(prob), **metricas}

externas_mlp = pd.DataFrame([
    evaluar_modelo_externo("red_neuronal_mlp", pipelines_originales["red_neuronal_mlp"], p)
    for p in ["VALIDACION", "PRUEBA"]
])
externas_mlp

,modelo,particion,observaciones,pr_auc,f05,precision,recall,fp,fn,tp,tn,costo_total_relativo,costo_por_1000,prevalencia
0,red_neuronal_mlp,VALIDACION,9798,0.039071,0.021067,0.017647,0.09375,334,58,6,9400,1494.0,152.480098,0.006532
1,red_neuronal_mlp,PRUEBA,9782,0.013095,0.010020,0.008180,0.10000,485,36,4,9257,1205.0,123.185443,0.004089


### 8.3. Evaluar Gradient Boosting entrenado

In [35]:
#? Medir con los mismos costos, umbral y observaciones.
externas_boosting = pd.DataFrame([
    evaluar_modelo_externo("gradient_boosting", pipelines_originales["gradient_boosting"], p)
    for p in ["VALIDACION", "PRUEBA"]
])
externas_boosting

,modelo,particion,observaciones,pr_auc,f05,precision,recall,fp,fn,tp,tn,costo_total_relativo,costo_por_1000,prevalencia
0,gradient_boosting,VALIDACION,9798,0.023344,0.048343,0.042424,0.109375,158,57,7,9576,1298.0,132.476016,0.006532
1,gradient_boosting,PRUEBA,9782,0.025542,0.035545,0.029851,0.150000,195,34,6,9547,875.0,89.450010,0.004089


### 8.4. Guardar la comparación de costos

In [36]:
#? Comparar FN, FP, PR-AUC, F0.5 y costo por 1000.
costos_externos = pd.concat([externas_mlp, externas_boosting], ignore_index=True)
RUTA_COSTOS = DIRECTORIO / "validacion_costos_proyecto.csv"
costos_externos.to_csv(RUTA_COSTOS, index=False)
costos_externos

,modelo,particion,observaciones,pr_auc,f05,precision,recall,fp,fn,tp,tn,costo_total_relativo,costo_por_1000,prevalencia
0,red_neuronal_mlp,VALIDACION,9798,0.039071,0.021067,0.017647,0.093750,334,58,6,9400,1494.0,152.480098,0.006532
1,red_neuronal_mlp,PRUEBA,9782,0.013095,0.010020,0.008180,0.100000,485,36,4,9257,1205.0,123.185443,0.004089
2,gradient_boosting,VALIDACION,9798,0.023344,0.048343,0.042424,0.109375,158,57,7,9576,1298.0,132.476016,0.006532
3,gradient_boosting,PRUEBA,9782,0.025542,0.035545,0.029851,0.150000,195,34,6,9547,875.0,89.450010,0.004089


In [38]:
#? Comparación numérica de promedios de PR-AUC por esquema.
def mejor_por_esquema(resumen, esquema):
    return (resumen.loc[resumen["esquema"] == esquema]
            .sort_values("pr_auc_promedio", ascending=False).iloc[0])

m_temporal = mejor_por_esquema(resumen_cv, "temporal")
m_aleatorio = mejor_por_esquema(resumen_cv, "aleatorio")

respuesta_2 = (
    f"En validación temporal, el mayor PR-AUC promedio fue "
    f"{m_temporal['pr_auc_promedio']:.6f} (desviación {m_temporal['pr_auc_desviacion']:.6f}) "
    f"para {m_temporal['modelo']}. En validación aleatoria, el mayor promedio fue "
    f"{m_aleatorio['pr_auc_promedio']:.6f} (desviación {m_aleatorio['pr_auc_desviacion']:.6f}) "
    f"para {m_aleatorio['modelo']}. "
    + ("La preferencia por modelo cambió entre esquemas. "
       if m_temporal['modelo'] != m_aleatorio['modelo']
       else "La preferencia por modelo fue la misma en ambos esquemas. ")
    + "Las diferencias también pueden reflejar cambios en la prevalencia por fecha, "
      "ya que la muestra de entrenamiento fue balanceada parcialmente."
)
print(respuesta_2)

En validación temporal, el mayor PR-AUC promedio fue 0.472294 (desviación 0.231850) para gradient_boosting. En validación aleatoria, el mayor promedio fue 0.561057 (desviación 0.013966) para gradient_boosting. La preferencia por modelo fue la misma en ambos esquemas. Las diferencias también pueden reflejar cambios en la prevalencia por fecha, ya que la muestra de entrenamiento fue balanceada parcialmente.


## 10. Comprobación del entregable

Al finalizar se generan tres CSV: `validacion_pliegues_proyecto.csv` con las **12 evaluaciones**, `validacion_resumen_proyecto.csv` con **promedio y desviación**, y `validacion_costos_proyecto.csv` con la evaluación externa en validación y prueba. Los resultados dependen del CSV congelado, los `run_id` y las versiones del entorno.

La siguiente tabla queda disponible para revisar los archivos. Si vas a compartir el notebook sin la muestra, conserva por lo menos el manifiesto, los identificadores de MLflow y la huella SHA-256 para documentar exactamente qué datos se utilizaron.

In [41]:
#? Verificar las salidas y su tamaño sin abrir ventanas externas.
comprobacion = pd.DataFrame([
    {"archivo": p.name,
     "existe": p.exists(),
     "bytes": p.stat().st_size if p.exists() else 0}
    for p in [ARCHIVO_DATOS, ARCHIVO_MANIFIESTO,
              RUTA_PLIEGUES, RUTA_RESUMEN, RUTA_COSTOS]
])
comprobacion

,archivo,existe,bytes
0,mc1_muestra_experimentos.csv,True,13911975
1,mc1_manifiesto_experimento.json,True,3832
2,validacion_pliegues_proyecto.csv,True,2725
3,validacion_resumen_proyecto.csv,True,811
4,validacion_costos_proyecto.csv,True,756
